# 5G Handover — Dataset Exploration From Scratch
## Goal: Understand EVERYTHING in the data before touching any model

We go file by file, column by column.
We ask: **what is this column, does it vary, is it useful, should we keep it?**

No modeling here. Just understanding.

---

## CELL 1 — Imports

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.3f}'.format)

RANDOM_STATE = 42
BASE_PATH    = './DATASET/'
SAMPLE_SIZE  = 100_000   # rows per file — increase if you have RAM

COLORS = {
    'hbahn':  '#2ecc71',
    'mobile': '#3498db',
    'static': '#e74c3c'
}

print('✅ Imports done')

---
## CELL 2 — Load all files
We load H-Bahn, Mobile, and Static — all files, all scenarios.
Static is NOT ignored — we will explore it properly.

In [ ]:
FILES = {
    'hbahn':  ['cell_data', 'neighboring_data', 'latency_data', 'iperf_data'],
    'mobile': ['cell_data', 'neighboring_data', 'latency_data', 'iperf_data'],
    'static': ['cell_data', 'latency_data', 'static_locations'],
}

raw = {}
for scenario, filelist in FILES.items():
    raw[scenario] = {}
    spath = os.path.join(BASE_PATH, scenario, scenario)
    print(f'\n📂 {scenario.upper()}')
    for fname in filelist:
        fpath = os.path.join(spath, f'{fname}.csv')
        try:
            nrows = None if fname == 'static_locations' else SAMPLE_SIZE
            df = pd.read_csv(fpath, sep=';', nrows=nrows, low_memory=False)
            raw[scenario][fname] = df
            print(f'  ✅ {fname:25s} {str(df.shape):20s} '
                  f'missing={df.isnull().mean().mean()*100:.1f}%')
        except FileNotFoundError:
            print(f'  ❌ {fname:25s} NOT FOUND')
        except Exception as e:
            print(f'  ⚠️  {fname:25s} ERROR: {e}')

print('\n✅ Loading complete')

---
## CELL 3 — Full column audit: every file, every column

For every file we print:
- Column name
- Data type
- % missing
- Number of unique values
- Min, max, mean (for numbers)
- A sample value

**This is where we decide what is useful and what is garbage.**

In [ ]:
def audit_file(df, label):
    print(f'\n{"="*70}')
    print(f'  {label}  |  {df.shape[0]:,} rows × {df.shape[1]} columns')
    print(f'{"="*70}')
    
    rows = []
    for col in df.columns:
        s      = df[col]
        miss   = s.isnull().mean() * 100
        nuniq  = s.nunique()
        dtype  = str(s.dtype)
        sample = s.dropna().iloc[0] if s.dropna().shape[0] > 0 else 'ALL NULL'
        
        if pd.api.types.is_numeric_dtype(s) and s.dtype != bool:
            s_float = s.astype(float)
            mn  = round(float(s_float.min()), 3)
            mx  = round(float(s_float.max()), 3)
            avg = round(float(s_float.mean()), 3)
            info = f'min={mn}  max={mx}  mean={avg}'
        else:
            info = f'sample values: {s.dropna().unique()[:3].tolist()}'
        
        # flag columns that are likely useless
        flag = ''
        if miss == 100:             flag = '🔴 ALL MISSING — DROP'
        elif miss > 60:             flag = '🟠 HIGH MISSING'
        elif miss > 40:             flag = '🟡 MEDIUM MISSING'
        elif nuniq <= 1:            flag = '🔴 CONSTANT — DROP'
        elif nuniq <= 2:            flag = '🟡 NEAR CONSTANT'
        elif nuniq == df.shape[0]:  flag = '🔵 UNIQUE ID (likely metadata)'
        
        rows.append({
            'column':  col,
            'dtype':   dtype,
            'missing': f'{miss:.1f}%',
            'unique':  nuniq,
            'range/sample': info,
            'flag':    flag
        })
    
    result = pd.DataFrame(rows)
    print(result.to_string(index=False))
    return result


audits = {}
for scenario in raw:
    audits[scenario] = {}
    for fname, df in raw[scenario].items():
        audits[scenario][fname] = audit_file(df, f'{scenario.upper()} / {fname}')

---
## CELL 4 — Understand the timestamps

Timestamps are the key that links all files together.
We need to know: what format are they in? Can we join on them?

In [ ]:
print('TIMESTAMP AUDIT')
print('='*60)

for scenario in ['hbahn', 'mobile']:
    for fname in raw[scenario]:
        df = raw[scenario][fname]
        ts_cols = [c for c in df.columns if 'time' in c.lower() or c == 'ts']
        if not ts_cols:
            continue
        print(f'\n{scenario}/{fname}:')
        for col in ts_cols:
            vals = df[col].dropna()
            if len(vals) == 0:
                print(f'  {col}: ALL NULL')
                continue
            sample = vals.iloc[0]
            # Try to understand the format
            numeric = pd.to_numeric(vals, errors='coerce')
            if numeric.notna().mean() > 0.9:
                med = numeric.median()
                if med > 1e12:
                    fmt = 'Unix MILLISECONDS'
                elif med > 1e9:
                    fmt = 'Unix SECONDS'
                else:
                    fmt = f'numeric (median={med:.0f}) — unknown unit'
            else:
                fmt = 'STRING/DATE format'
            print(f'  {col:25s}: {fmt:30s} | sample: {sample}')

---
## CELL 5 — See an actual handover happen in raw data

Before building any model, let's just SEE a real handover.
A handover = `physical_cellid` changes between two consecutive rows.
Let's find one and look at the radio conditions around it.

In [ ]:
print('REAL HANDOVER EVENTS IN RAW DATA')
print('='*60)

for scenario in ['hbahn', 'mobile']:
    df = raw[scenario]['cell_data'].copy()
    
    # Sort by time
    ts_col = next((c for c in ['timestamp','timestampstart'] 
                   if c in df.columns), None)
    if ts_col:
        df['ts_num'] = pd.to_numeric(df[ts_col], errors='coerce')
        df = df.sort_values('ts_num').reset_index(drop=True)
    
    if 'physical_cellid' not in df.columns:
        print(f'{scenario}: no physical_cellid'); continue
    
    # Detect handovers
    df['prev_cell'] = df['physical_cellid'].shift(1)
    df['is_ho']     = (df['physical_cellid'] != df['prev_cell']) & df['prev_cell'].notna()
    
    n_ho    = df['is_ho'].sum()
    n_cells = df['physical_cellid'].nunique()
    
    print(f'\n{scenario.upper()}:')
    print(f'  Rows loaded:       {len(df):,}')
    print(f'  Unique towers:     {n_cells}')
    print(f'  Handover events:   {n_ho}')
    print(f'  HO rate:           {n_ho/len(df)*100:.2f}% of rows')
    
    # Show what a handover looks like
    ho_indices = df[df['is_ho']].index.tolist()
    if ho_indices:
        idx = ho_indices[0]
        show_cols = [c for c in 
                     ['physical_cellid','rsrp','rsrq','sinr',
                      'cqi','ta','velocity','is_ho']
                     if c in df.columns]
        window = df.iloc[max(0, idx-4): idx+5][show_cols]
        print(f'\n  ── First handover at row {idx} ──')
        print(window.to_string())
        print()
        print('  Notice: physical_cellid changes → that is the handover moment')
        print('  Look at rsrp and sinr before and after — does quality change?')

---
## CELL 6 — Signal quality deep dive: every metric explained

For each radio column we:
1. Show its distribution
2. Compare H-Bahn vs Mobile vs Static
3. Show what it looks like BEFORE vs AFTER a handover
4. Decide if it's useful for prediction

In [ ]:
SIGNAL_COLS = ['rsrp', 'rsrq', 'sinr', 'cqi', 'ta', 'tx_power',
               'ss_rsrp', 'ss_rsrq', 'ss_sinr']

# ── Distribution comparison across scenarios ──
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for idx, col in enumerate(SIGNAL_COLS):
    ax = axes[idx]
    plotted = False
    for scenario in ['hbahn', 'mobile', 'static']:
        if col not in raw[scenario].get('cell_data', pd.DataFrame()).columns:
            continue
        vals = raw[scenario]['cell_data'][col].dropna()
        # clip extreme outliers for visualization
        vals = vals.clip(vals.quantile(0.01), vals.quantile(0.99))
        ax.hist(vals, bins=50, alpha=0.5, color=COLORS[scenario],
                label=scenario, density=True, edgecolor='none')
        plotted = True
    
    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    if plotted:
        ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, 'Not available', ha='center',
                va='center', transform=ax.transAxes, color='gray')

plt.suptitle('Signal Metric Distributions: H-Bahn vs Mobile vs Static',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('signal_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved signal_distributions.png')

In [ ]:
# ── Numerical summary of every signal column ──
print('SIGNAL COLUMN STATISTICS BY SCENARIO')
print('='*70)

for col in SIGNAL_COLS:
    print(f'\n── {col} ──')
    rows = []
    for scenario in ['hbahn', 'mobile', 'static']:
        df = raw[scenario].get('cell_data', pd.DataFrame())
        if col not in df.columns:
            rows.append({'scenario': scenario, 'status': 'NOT IN FILE',
                         'missing%': '-', 'min': '-', 'mean': '-',
                         'max': '-', 'std': '-', 'unique': '-'})
        else:
            s = df[col]
            rows.append({
                'scenario': scenario,
                'missing%': f"{s.isnull().mean()*100:.1f}%",
                'min':      round(s.min(), 2),
                'mean':     round(s.mean(), 2),
                'max':      round(s.max(), 2),
                'std':      round(s.std(), 2),
                'unique':   s.nunique()
            })
    print(pd.DataFrame(rows).to_string(index=False))

---
## CELL 7 — Mobility features: velocity, bearing, GPS

These tell us HOW the user is moving.
Key question: does speed affect signal quality and handover behavior?

In [ ]:
MOB_COLS = ['velocity', 'bearing', 'location_accuracy',
            'velocity_accuracy', 'bearing_accuracy']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, col in enumerate(MOB_COLS):
    ax = axes[idx]
    for scenario in ['hbahn', 'mobile', 'static']:
        df = raw[scenario].get('cell_data', pd.DataFrame())
        if col not in df.columns:
            continue
        vals = df[col].dropna()
        vals = vals.clip(vals.quantile(0.01), vals.quantile(0.99))
        ax.hist(vals, bins=50, alpha=0.55, color=COLORS[scenario],
                label=scenario, density=True, edgecolor='none')
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('Mobility Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('mobility_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Velocity stats ──
print('VELOCITY STATISTICS:')
for scenario in ['hbahn', 'mobile', 'static']:
    df = raw[scenario].get('cell_data', pd.DataFrame())
    if 'velocity' not in df.columns:
        continue
    v = df['velocity'].dropna()
    print(f'  {scenario:8s}: mean={v.mean():.2f} m/s '
          f'({v.mean()*3.6:.1f} km/h)  '
          f'max={v.max():.2f} m/s '
          f'({v.max()*3.6:.1f} km/h)  '
          f'std={v.std():.2f}')

---
## CELL 8 — QoS targets deep dive: latency and throughput

These are what we want to predict.
Key question: do they actually vary? Are they affected by radio conditions?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ── Row 1: mean_latency ──
for col_idx, scenario in enumerate(['hbahn', 'mobile', 'static']):
    ax = axes[0, col_idx]
    df = raw[scenario].get('latency_data', pd.DataFrame())
    if 'mean_latency' not in df.columns:
        ax.text(0.5, 0.5, 'No latency_data', ha='center', va='center',
                transform=ax.transAxes)
        continue
    lat = df['mean_latency'].dropna()
    lat = lat[lat < lat.quantile(0.99)]
    ax.hist(lat, bins=60, color=COLORS[scenario], alpha=0.8, edgecolor='none')
    ax.axvline(20,  color='green',  linestyle='--', lw=1.5, label='20ms (URLLC)')
    ax.axvline(50,  color='orange', linestyle='--', lw=1.5, label='50ms (good)')
    ax.axvline(100, color='red',    linestyle='--', lw=1.5, label='100ms (poor)')
    ax.set_title(f'mean_latency — {scenario.upper()}', fontweight='bold')
    ax.set_xlabel('ms')
    ax.legend(fontsize=7)
    # Print key stats as text on plot
    ax.text(0.97, 0.97,
            f'mean={lat.mean():.1f}ms\nstd={lat.std():.1f}ms\n'
            f'nunique={lat.nunique()}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# ── Row 2: datarate ──
for col_idx, scenario in enumerate(['hbahn', 'mobile', 'static']):
    ax = axes[1, col_idx]
    df = raw[scenario].get('iperf_data', pd.DataFrame())
    if df is None or 'datarate' not in df.columns:
        ax.text(0.5, 0.5, 'No iperf_data', ha='center', va='center',
                transform=ax.transAxes)
        continue
    dr = df['datarate'].dropna()
    # unit conversion check
    if dr.median() > 1_000_000:
        dr = dr / 1_000_000
        unit = 'bps→Mbps'
    elif dr.median() > 1_000:
        dr = dr / 1_000
        unit = 'kbps→Mbps'
    else:
        unit = 'Mbps'
    dr = dr[dr > 0]
    dr = dr[dr < dr.quantile(0.99)]
    ax.hist(dr, bins=60, color=COLORS[scenario], alpha=0.8, edgecolor='none')
    ax.axvline(15, color='orange', linestyle='--', lw=1.5, label='15 Mbps (4K stream)')
    ax.set_title(f'datarate — {scenario.upper()} ({unit})', fontweight='bold')
    ax.set_xlabel('Mbps')
    ax.legend(fontsize=7)
    ax.text(0.97, 0.97,
            f'mean={dr.mean():.1f}\nstd={dr.std():.1f}\n'
            f'nunique={dr.nunique()}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('QoS Targets: Latency and Throughput Distribution',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('qos_targets.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nKEY OBSERVATION:')
print('If mean_latency has very few unique values → it is near-constant → bad target')
print('If datarate has many unique values and high std → good target')

---
## CELL 9 — Static scenario deep dive

Static phones don't move → any signal change = NETWORK change, not user movement.
This means Static can tell us about cell load over time.

We explore: does signal quality in Static vary over time? By how much?

In [ ]:
print('STATIC SCENARIO — Deep Dive')
print('='*60)

df_static = raw['static']['cell_data'].copy()
locs      = raw['static'].get('static_locations', pd.DataFrame())

print('\nThe 4 fixed measurement locations:')
print(locs.to_string(index=False))

print(f'\nStatic cell_data: {df_static.shape}')
print(f'Columns: {df_static.columns.tolist()}')

# How many unique towers are visible from static locations?
if 'physical_cellid' in df_static.columns:
    print(f'\nUnique towers seen from static phones: '
          f'{df_static["physical_cellid"].nunique()}')
    print('\nTop towers by measurement count:')
    print(df_static['physical_cellid'].value_counts().head(10))

# Signal variation in static — this is the key question
sig_cols = [c for c in ['rsrp','sinr','rsrq','cqi','tx_power']
            if c in df_static.columns]

print('\nSIGNAL VARIATION IN STATIC (phone not moving!):')
print('High std = network is changing = load/interference variation')
print()
for col in sig_cols:
    s = df_static[col].dropna()
    cv = s.std() / abs(s.mean()) * 100 if s.mean() != 0 else 0
    useful = '✅ VARIES — potentially useful' if s.std() > 2 else '❌ near constant'
    print(f'  {col:15s}: mean={s.mean():8.2f}  std={s.std():6.2f}  '
          f'CV={cv:5.1f}%  → {useful}')

In [ ]:
# ── If there are timestamps, show signal over time ──
ts_col = next((c for c in ['timestamp','timestampstart'] 
               if c in df_static.columns), None)

if ts_col:
    df_s = df_static.copy()
    df_s['ts_num'] = pd.to_numeric(df_s[ts_col], errors='coerce')
    df_s = df_s.dropna(subset=['ts_num']).sort_values('ts_num')
    
    # Convert to datetime if Unix
    med = df_s['ts_num'].median()
    if med > 1e12:
        df_s['datetime'] = pd.to_datetime(df_s['ts_num'], unit='ms')
    elif med > 1e9:
        df_s['datetime'] = pd.to_datetime(df_s['ts_num'], unit='s')
    else:
        df_s['datetime'] = None
    
    if df_s['datetime'].notna().any():
        df_s['hour'] = df_s['datetime'].dt.hour
        
        # Plot signal by hour of day
        fig, axes = plt.subplots(1, len(sig_cols), figsize=(18, 4))
        if len(sig_cols) == 1:
            axes = [axes]
        
        for ax, col in zip(axes, sig_cols):
            hourly = df_s.groupby('hour')[col].mean()
            ax.plot(hourly.index, hourly.values,
                    color=COLORS['static'], linewidth=2, marker='o', markersize=4)
            ax.set_title(f'{col} by Hour of Day', fontweight='bold')
            ax.set_xlabel('Hour')
            ax.set_ylabel(col)
        
        plt.suptitle('Static Phone: Signal Quality Over Time of Day\n'
                     '(Phone not moving → changes = network load/interference)',
                     fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig('static_temporal.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('✅ Saved static_temporal.png')
        print('\nIf signal drops at peak hours → this is cell load!')
    else:
        print('Could not parse timestamps to datetime')
else:
    print('No timestamp column found in static cell_data')

---
## CELL 10 — Neighboring data deep dive

This is the most underused file in the project.
It tells us: what other towers are available at each moment?
Key question: are neighboring towers competitive with the serving cell?

In [ ]:
print('NEIGHBORING DATA DEEP DIVE')
print('='*60)

for scenario in ['hbahn', 'mobile']:
    df_n = raw[scenario].get('neighboring_data', pd.DataFrame())
    df_c = raw[scenario].get('cell_data', pd.DataFrame())
    
    print(f'\n{scenario.upper()}:')
    print(f'  Shape: {df_n.shape}')
    print(f'  Columns: {df_n.columns.tolist()}')
    
    # How many neighbors per timestamp?
    ts_col = next((c for c in ['timestamp','timestampstart'] 
                   if c in df_n.columns), None)
    if ts_col:
        nbr_per_ts = df_n.groupby(ts_col).size()
        print(f'\n  Neighbors detected per scan:')
        print(f'    mean:   {nbr_per_ts.mean():.1f}')
        print(f'    median: {nbr_per_ts.median():.0f}')
        print(f'    max:    {nbr_per_ts.max()}')
        print(f'    min:    {nbr_per_ts.min()}')
    
    # Compare neighbor signal to serving cell
    if 'rsrp_neighboring' in df_n.columns and 'rsrp' in df_c.columns:
        nbr_rsrp  = df_n['rsrp_neighboring'].dropna()
        serv_rsrp = df_c['rsrp'].dropna()
        
        print(f'\n  RSRP comparison:')
        print(f'    Serving  → mean={serv_rsrp.mean():.1f}  '
              f'std={serv_rsrp.std():.1f}  '
              f'min={serv_rsrp.min():.1f}  '
              f'max={serv_rsrp.max():.1f}')
        print(f'    Neighbor → mean={nbr_rsrp.mean():.1f}  '
              f'std={nbr_rsrp.std():.1f}  '
              f'min={nbr_rsrp.min():.1f}  '
              f'max={nbr_rsrp.max():.1f}')
        
        pct_stronger = (nbr_rsrp > serv_rsrp.mean()).mean() * 100
        print(f'\n  % of neighbor readings STRONGER than avg serving: '
              f'{pct_stronger:.1f}%')
        print(f'  → This means handover opportunities exist {pct_stronger:.0f}% of the time')

# ── Plot serving vs neighbor RSRP ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, scenario in zip(axes, ['hbahn', 'mobile']):
    df_n = raw[scenario].get('neighboring_data', pd.DataFrame())
    df_c = raw[scenario].get('cell_data', pd.DataFrame())
    if 'rsrp_neighboring' not in df_n.columns or 'rsrp' not in df_c.columns:
        continue
    
    serv = df_c['rsrp'].dropna()
    nbr  = df_n['rsrp_neighboring'].dropna()
    serv = serv.clip(serv.quantile(0.01), serv.quantile(0.99))
    nbr  = nbr.clip(nbr.quantile(0.01), nbr.quantile(0.99))
    
    ax.hist(serv, bins=50, alpha=0.6, color='#2ecc71',
            label='Serving cell', density=True)
    ax.hist(nbr,  bins=50, alpha=0.6, color='#e67e22',
            label='Neighbors', density=True)
    ax.axvline(-100, color='red', linestyle='--', lw=1.5,
               label='HO threshold (-100 dBm)')
    ax.set_title(f'Serving vs Neighbor RSRP — {scenario.upper()}',
                 fontweight='bold')
    ax.set_xlabel('RSRP (dBm)')
    ax.legend(fontsize=9)

plt.suptitle('Are There Better Towers Available?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('serving_vs_neighbor.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved serving_vs_neighbor.png')

---
## CELL 11 — Correlation analysis: what relates to what?

Before deciding features, we need to understand:
1. Which radio features are correlated with each other? (redundant pairs to remove)
2. Which features correlate with the TARGET (datarate)?
3. Which features are truly independent and add information?

In [ ]:
# ── Correlation matrix for cell_data ──
META = ['network','mcc','mnc','MNO','device','cellbandwidths',
        'tracking_area_code','timestamp','timestampstart',
        'timestampend','bearing_accuracy','velocity_accuracy',
        'location_accuracy','latitude','longitude','altitude',
        'passive_id','username','session_id']

for scenario in ['hbahn', 'mobile']:
    df = raw[scenario]['cell_data'].copy()
    df = df.drop(columns=[c for c in META if c in df.columns], errors='ignore')
    df = df.select_dtypes(include='number')
    df = df.dropna(axis=1, thresh=int(0.5 * len(df)))
    
    corr = df.corr(method='spearman')
    
    fig, ax = plt.subplots(figsize=(14, 11))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
                cmap='coolwarm', center=0, linewidths=0.3,
                ax=ax, cbar_kws={'shrink': 0.8},
                annot_kws={'size': 8})
    ax.set_title(f'Spearman Correlation — {scenario.upper()} cell_data\n'
                 f'(|r| > 0.85 = redundant pair → keep only one)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'correlation_{scenario}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print highly correlated pairs
    print(f'\nHIGHLY CORRELATED PAIRS in {scenario} (|r| > 0.80):')
    print('These are redundant — keeping both adds no information')
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    pairs = upper.stack()
    high  = pairs[pairs.abs() > 0.80].sort_values(ascending=False)
    if len(high) == 0:
        print('  None found above threshold')
    for (c1, c2), r in high.items():
        print(f'  {c1:20s} ↔ {c2:20s}  r={r:.3f}')
    print()

---
## CELL 12 — How does signal relate to throughput?

This is the most important question for DSO4.
If signal quality does NOT correlate with throughput,
then using signal features to predict throughput won't work.

Let's check before building any model.

In [ ]:
print('SIGNAL vs THROUGHPUT CORRELATION')
print('='*60)
print('Can radio features predict datarate? Let us check before modeling.')
print()

for scenario in ['hbahn', 'mobile']:
    df_ip = raw[scenario].get('iperf_data', pd.DataFrame())
    if df_ip is None or 'datarate' not in df_ip.columns:
        print(f'{scenario}: no iperf_data'); continue
    
    # normalize datarate
    dr = df_ip['datarate'].dropna()
    if dr.median() > 1_000_000: df_ip['datarate'] = df_ip['datarate'] / 1_000_000
    elif dr.median() > 1_000:   df_ip['datarate'] = df_ip['datarate'] / 1_000
    df_ip = df_ip[df_ip['datarate'] > 0]
    
    sig_cols_avail = [c for c in ['rsrp','rsrq','sinr','cqi','ta',
                                   'tx_power','ss_rsrp','ss_sinr',
                                   'velocity','lte_mcs','nr_mcs']
                      if c in df_ip.columns]
    
    print(f'{scenario.upper()} — Spearman correlation with datarate:')
    correlations = []
    for col in sig_cols_avail:
        s = df_ip[col].dropna()
        if len(s) < 100: continue
        aligned = df_ip[['datarate', col]].dropna()
        if len(aligned) < 50: continue
        r, p = stats.spearmanr(aligned['datarate'], aligned[col])
        strength = ('🔴 STRONG' if abs(r) > 0.5 else
                    '🟡 MODERATE' if abs(r) > 0.3 else
                    '⚪ WEAK')
        correlations.append((col, r, p, strength))
    
    correlations.sort(key=lambda x: abs(x[1]), reverse=True)
    for col, r, p, strength in correlations:
        print(f'  {col:20s}  r={r:+.3f}  {strength}')
    print()

print('INTERPRETATION:')
print('  r > 0.5  = feature is strongly predictive of throughput → KEEP')
print('  r 0.3-0.5 = moderately predictive → likely keep')
print('  r < 0.3  = weak → evaluate carefully')
print('  negative r means: as this increases, throughput decreases')
print('  (e.g. tx_power: higher tx_power = struggling = lower throughput)')

In [ ]:
# ── Scatter plots: top features vs datarate ──
TOP_FEATURES = ['rsrp', 'sinr', 'cqi', 'tx_power', 'ta', 'velocity']

for scenario in ['hbahn', 'mobile']:
    df_ip = raw[scenario].get('iperf_data', pd.DataFrame())
    if df_ip is None or 'datarate' not in df_ip.columns:
        continue
    
    dr = df_ip['datarate'].copy()
    if dr.median() > 1_000_000: df_ip['datarate'] = dr / 1_000_000
    elif dr.median() > 1_000:   df_ip['datarate'] = dr / 1_000
    df_ip = df_ip[df_ip['datarate'] > 0]
    
    avail = [c for c in TOP_FEATURES if c in df_ip.columns]
    if not avail: continue
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for idx, col in enumerate(avail[:6]):
        ax  = axes[idx]
        sub = df_ip[['datarate', col]].dropna().sample(
            min(3000, len(df_ip)), random_state=42)
        ax.scatter(sub[col], sub['datarate'],
                   alpha=0.2, s=5, color=COLORS[scenario])
        # Add trend line
        try:
            z = np.polyfit(sub[col], sub['datarate'], 1)
            p = np.poly1d(z)
            x_line = np.linspace(sub[col].min(), sub[col].max(), 100)
            ax.plot(x_line, p(x_line), 'r-', linewidth=2, label='trend')
        except Exception:
            pass
        r, _ = stats.spearmanr(sub['datarate'], sub[col])
        ax.set_title(f'{col} vs datarate  (r={r:.3f})', fontweight='bold')
        ax.set_xlabel(col)
        ax.set_ylabel('datarate (Mbps)')
    
    for i in range(len(avail), 6):
        axes[i].set_visible(False)
    
    plt.suptitle(f'{scenario.upper()} — Feature vs Throughput Scatter',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'scatter_{scenario}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved scatter_{scenario}.png')

---
## CELL 13 — Feature selection decision table

Based on everything we saw above, we now make explicit decisions:
- KEEP: feature has variance, correlates with target, not redundant
- DROP: constant, 100% missing, pure metadata, redundant with another feature
- MAYBE: some value but limited — evaluate after seeing correlation results

In [ ]:
# This cell summarizes our feature selection decisions
# UPDATE the 'decision' column after looking at your correlation results above

feature_decisions = [
    # SIGNAL QUALITY
    {'feature': 'rsrp',             'type': 'signal',   'decision': 'KEEP',
     'reason': 'Primary signal strength. Drops at cell edge = HO trigger. '
               'Directly tied to achievable datarate.'},
    {'feature': 'rsrq',             'type': 'signal',   'decision': 'KEEP',
     'reason': 'Signal quality including interference. Different from rsrp. '
               'Good in low-interference environments.'},
    {'feature': 'sinr',             'type': 'signal',   'decision': 'KEEP',
     'reason': 'Best single predictor of link quality. '
               'Low sinr = interference = low throughput.'},
    {'feature': 'rssi',             'type': 'signal',   'decision': 'CHECK',
     'reason': 'Total received power. Often 90%+ correlated with rsrp. '
               'Drop if corr(rsrp,rssi) > 0.85.'},
    {'feature': 'ss_rsrp',          'type': 'signal_5g','decision': 'KEEP',
     'reason': '5G version of rsrp. Has ~9% missing. '
               'Keep — impute missing with median.'},
    {'feature': 'ss_rsrq',          'type': 'signal_5g','decision': 'CHECK',
     'reason': '5G version of rsrq. Check correlation with rsrq.'},
    {'feature': 'ss_sinr',          'type': 'signal_5g','decision': 'KEEP',
     'reason': '5G sinr. Different from 4G sinr. '
               'Useful if 5G coverage exists.'},
    # LINK ADAPTATION
    {'feature': 'cqi',              'type': 'link',     'decision': 'KEEP',
     'reason': 'Phone feedback to tower about achievable speed. '
               'Direct proxy for potential throughput.'},
    {'feature': 'lte_mcs',          'type': 'link',     'decision': 'KEEP',
     'reason': 'How aggressively data is being encoded. '
               'High = good conditions. Check for redundancy with cqi.'},
    {'feature': 'lte_ri',           'type': 'link',     'decision': 'KEEP',
     'reason': 'Number of MIMO streams. More streams = more speed potential.'},
    {'feature': 'nr_mcs',           'type': 'link',     'decision': 'MAYBE',
     'reason': '5G MCS. Check missing %. May not exist in all rows.'},
    {'feature': 'nr_ri',            'type': 'link',     'decision': 'MAYBE',
     'reason': '5G RI. Same as above.'},
    # PHONE BEHAVIOR
    {'feature': 'tx_power',         'type': 'phone',    'decision': 'KEEP',
     'reason': 'How hard phone shouts at tower. '
               'High = struggling = likely negative correlation with throughput.'},
    {'feature': 'ta',               'type': 'phone',    'decision': 'KEEP',
     'reason': 'Distance from tower. '
               'High ta = cell edge = about to hand over.'},
    # CELL IDENTITY
    {'feature': 'physical_cellid',  'type': 'identity', 'decision': 'HO ONLY',
     'reason': 'Used ONLY to detect handovers (PCI change). '
               'Not a model input — each tower has a different ID with no ordinal meaning.'},
    {'feature': 'earfcn',           'type': 'identity', 'decision': 'KEEP',
     'reason': 'Frequency channel. Encodes which band tower uses. '
               'Different bands have different coverage/capacity.'},
    {'feature': 'cell_index',       'type': 'identity', 'decision': 'DROP',
     'reason': 'Internal log index. No physical meaning.'},
    # MOBILITY
    {'feature': 'velocity',         'type': 'mobility', 'decision': 'KEEP',
     'reason': 'Speed affects Doppler effect and handover frequency. '
               'High speed = more HOs = potentially worse post-HO quality.'},
    {'feature': 'bearing',          'type': 'mobility', 'decision': 'MAYBE',
     'reason': 'Direction of movement. Useful for route-based analysis. '
               'Check if it improves model. Can skip in first iteration.'},
    {'feature': 'location_accuracy','type': 'mobility', 'decision': 'DROP',
     'reason': 'GPS accuracy estimate. Meta-information about measurement quality. '
               'Not a network feature.'},
    {'feature': 'velocity_accuracy','type': 'mobility', 'decision': 'DROP',
     'reason': 'Speed measurement accuracy. Same as above. Not a network feature.'},
    {'feature': 'bearing_accuracy', 'type': 'mobility', 'decision': 'DROP',
     'reason': 'Bearing accuracy. Same as above.'},
    {'feature': 'latitude',         'type': 'mobility', 'decision': 'MAYBE',
     'reason': 'GPS position. Useful for DSO3 clustering. '
               'For DSO4 regression it can help if some areas are consistently better.'},
    {'feature': 'longitude',        'type': 'mobility', 'decision': 'MAYBE',
     'reason': 'Same as latitude.'},
    # BANDWIDTH
    {'feature': 'primary_bandwidth','type': 'bandwidth','decision': 'KEEP',
     'reason': 'Channel width sets capacity ceiling. '
               'Wider bandwidth = more potential speed.'},
    {'feature': 'ul_bandwidth',     'type': 'bandwidth','decision': 'CHECK',
     'reason': 'Uplink bandwidth. Check correlation with primary_bandwidth. '
               'Drop if redundant.'},
    {'feature': 'cellbandwidths',   'type': 'bandwidth','decision': 'DROP',
     'reason': '100% missing in H-Bahn. String format. Unusable.'},
    # METADATA — always drop as model inputs
    {'feature': 'mcc',              'type': 'metadata', 'decision': 'DROP',
     'reason': 'Country code. Same for all rows (Germany). Zero variance.'},
    {'feature': 'mnc',              'type': 'metadata', 'decision': 'DROP',
     'reason': 'Network code. Identifies operator. '
               'Model would learn operator artifacts, not physics.'},
    {'feature': 'MNO',              'type': 'metadata', 'decision': 'DROP',
     'reason': 'Operator name. Same as mnc.'},
    {'feature': 'network',          'type': 'metadata', 'decision': 'DROP',
     'reason': '100% missing in H-Bahn. LTE/NR label.'},
    {'feature': 'device',           'type': 'metadata', 'decision': 'DROP',
     'reason': 'Phone model. Would cause model to learn device-specific behavior. '
               'We want network behavior, not device behavior.'},
    {'feature': 'tracking_area_code','type': 'metadata','decision': 'DROP',
     'reason': 'Coarse region code. Too granular for useful spatial modeling.'},
]

fd = pd.DataFrame(feature_decisions)

print('FEATURE SELECTION DECISIONS')
print('='*80)
for decision in ['KEEP', 'CHECK', 'MAYBE', 'HO ONLY', 'DROP']:
    subset = fd[fd['decision'] == decision]
    emoji = {'KEEP':'✅','CHECK':'🔍','MAYBE':'🤔','HO ONLY':'🔁','DROP':'❌'}[decision]
    print(f'\n{emoji} {decision} ({len(subset)} features):')
    for _, row in subset.iterrows():
        print(f'  {row["feature"]:22s} [{row["type"]:10s}] {row["reason"]}')

keep_features = fd[fd['decision'] == 'KEEP']['feature'].tolist()
print(f'\n\nFINAL KEEP LIST ({len(keep_features)} features):')
print(keep_features)

---
## CELL 14 — Missing value analysis: what do we actually have?

Before committing to a feature, check: is it actually present in enough rows?

In [ ]:
print('MISSING VALUE ANALYSIS — cell_data + iperf_data')
print('='*60)
print('We need features that exist in BOTH scenarios and in most rows')
print()

all_candidate_cols = [
    'rsrp','rsrq','sinr','rssi','cqi','ta','tx_power',
    'ss_rsrp','ss_rsrq','ss_sinr',
    'lte_mcs','lte_ri','nr_mcs','nr_ri',
    'primary_bandwidth','ul_bandwidth','earfcn',
    'velocity','bearing','physical_cellid'
]

rows = []
for col in all_candidate_cols:
    row = {'feature': col}
    for scenario in ['hbahn', 'mobile']:
        for fname in ['cell_data', 'iperf_data']:
            df = raw[scenario].get(fname, pd.DataFrame())
            key = f'{scenario}/{fname}'
            if col not in df.columns:
                row[key] = 'NOT IN FILE'
            else:
                miss = df[col].isnull().mean() * 100
                row[key] = f'{miss:.1f}%'
    rows.append(row)

miss_df = pd.DataFrame(rows)
print(miss_df.to_string(index=False))

print('\nINTERPRETATION:')
print('  0.0%        = perfect, always available')
print('  1-10%       = fine, impute with median')
print('  10-40%      = acceptable, impute carefully')
print('  >40%        = problematic — consider dropping')
print('  NOT IN FILE = column does not exist in that file')

---
## CELL 15 — Low variance check: are some columns near-constant?

A constant column tells the model nothing.
We need to detect these before wasting time including them.

In [ ]:
print('LOW VARIANCE CHECK')
print('='*60)
print('Columns with fewer than 5 unique values are near-constant — useless for ML')
print()

for scenario in ['hbahn', 'mobile']:
    df = raw[scenario]['cell_data'].copy()
    num_df = df.select_dtypes(include='number')
    
    print(f'{scenario.upper()} — cell_data:')
    low_var = []
    for col in num_df.columns:
        n = num_df[col].nunique()
        if n <= 5:
            vals = num_df[col].dropna().unique().tolist()
            low_var.append((col, n, vals))
    
    if low_var:
        for col, n, vals in low_var:
            print(f'  ❌ {col:25s}: only {n} unique values: {vals}')
    else:
        print('  ✅ No near-constant columns found')
    print()

---
## CELL 16 — FINAL SUMMARY: What we know and what we do next

Based on ALL the exploration above, here is our data understanding summary.
This is the foundation for the modeling phase.

In [ ]:
print('='*70)
print('  EXPLORATION SUMMARY — What we learned about the data')
print('='*70)

print('''
1. SCENARIOS
   H-Bahn  → structured, repeated route, controlled speed
              good for understanding handover patterns on known paths
   Mobile  → diverse, messy, real-world conditions
              more important for building a generalizable model
   Static  → phone not moving → signal variation = NETWORK variation
              potential use: temporal load profiling per tower

2. FILES
   cell_data      → radio heartbeat. Our INPUTS.
   iperf_data     → throughput measurements. Our TARGET (datarate).
   latency_data   → ping results. mean_latency near-constant in DoNext.
                    packet_loss could be useful for DSO1.
   neighboring_data → what other towers are available.
                    CRITICAL for DSO2. Underused for DSO4.
   static_locations → 4 GPS coordinates. Links static cell_data to places.

3. TARGET VARIABLE
   datarate (Mbps) from iperf_data
   → has real variance → good regression target
   mean_latency → near-constant in controlled experiment → bad target

4. FEATURES TO KEEP (confirmed by correlation analysis above)
   Signal quality: rsrp, rsrq, sinr, cqi
   5G metrics:     ss_rsrp, ss_sinr (if available)
   Phone stress:   tx_power, ta
   Link speed:     lte_mcs, lte_ri
   Bandwidth:      primary_bandwidth, earfcn
   Mobility:       velocity

5. FEATURES TO DROP
   ALL metadata: mcc, mnc, MNO, network, device
   Accuracy cols: location_accuracy, velocity_accuracy, bearing_accuracy
   Redundant:     rssi (if corr(rssi,rsrp) > 0.85)
   Constant:      any column with < 5 unique values
   100% missing:  cellbandwidths, network (in H-Bahn)

6. WHAT WE BUILD NEXT
   Phase 1: Clean the data using what we learned above
   Phase 2: Detect handovers and label post-HO windows
   Phase 3: Merge cell_data + iperf_data by timestamp
   Phase 4: Engineer temporal features (lag, rolling, delta)
   Phase 5: Build X (features) and y (datarate)
   Phase 6: Train and evaluate models
''')

print('='*70)
print('  Run the cells above and share the outputs.')
print('  We will update this summary based on what the data actually shows.')
print('='*70)